# Week 12｜切塊策略與向量索引
**Chunking strategies, vector indexing and retrieval evaluation**

第 10、11 週我們比較了關鍵字檢索和語意檢索，也看到一個問題：**第 8 週切出來的 chunks 有些非常長（最長 14,350 字），句向量模型只讀得到前面一小段。**

切塊 (chunking) 是 RAG 系統中最影響檢索品質的步驟之一。今天比較四種切塊策略，用同一套問答集評估：

| | 策略 | 業界的名稱 |
|---|---|---|
| A | 第 8 週：在 `##` 標題處切 | 依標題切 (heading-based) |
| B | 每 N 個字切一刀，前後重疊 | **固定長度 + 重疊 (fixed-size with overlap)**：你今天要寫 |
| C | 依「第N條」切，太長再依「一、（一）」切 | 依文件結構切 (structure-based) + 遞迴切分 (recursive) |
| D | C ＋ 每塊開頭加【法規名稱｜章】 | **加上下文標題 (contextual chunk headers)** |

| 節次 | 時間 | 內容 | 任務 |
|---|---|---|---|
| 第 1 節 | 9:00–9:50 | 為什麼長 chunk 不好；四種切塊策略；寫出固定長度切塊 | 1-A 教師示範、1-B 兩人一組 |
| 第 2 節 | 10:00–10:50 | 建立向量索引；用問答集自動計算 hit@k | 2-A 教師示範、2-B 兩人一組 |
| 第 3 節 | 11:00–11:50 | 四種策略一起比較；自己出題；繳交 | 3-A 教師示範、3-C 個人 |

**今天要繳交（下課前 push）：**
1. `12_vector_indexing.ipynb`（保留輸出）
2. `week12_retrieval_eval.json`（最後一格自動產生）
3. `src/rechunk_v2.py`（老師提供，放進你的專案）
4. `README.md` 的 Week 12 進度、`AI_USAGE.md` 的 Week 12 段落、更新後的 `.gitignore`

標有 `# TODO` 的地方是你要完成的程式碼。每個 TODO 後面都有一格檢查，執行後出現 ✅ 才算完成。

**今天這份 notebook 的結果（向量索引）就是第 14 週問答系統要用的「檢索」部分。**

## 0. 開始之前

1. 從數位學習平台下載 `rechunk_v2.py`，放到專案的 **`src/`** 資料夾（沒有就新增一個）。
2. 確認 `data/interim/` 的三個子資料夾裡有第 8 週產生的 `raw_*.md`。**沒有的同學**：從數位學習平台下載 `week12_interim_raw.zip`，解壓縮到 `data/interim/`。
3. 這週**不需要安裝新套件**（沿用第 10 週的 `sentence-transformers`、`numpy`、`pandas`），模型沿用 `models/bge-small-zh-v1.5/`。

下一格檢查以上三件事。

In [1]:
import os, re, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd

notebook_dir = os.path.abspath('')
# 往上推兩層取得根目錄（notebook 放在 notebooks/week12/）
# project_root = os.path.dirname(os.path.dirname(notebook_dir))
# 往上推一層取得根目錄（notebook 放在 notebooks/）→ 把上一行加上 #，這一行的 # 拿掉
project_root = os.path.dirname(notebook_dir)
ROOT = Path(project_root)
print("專案根目錄：", ROOT)

assert (ROOT / "data").exists(), "找不到 data 資料夾：請檢查上面是往上推兩層還是一層"
assert (ROOT / "src" / "rechunk_v2.py").exists(), "找不到 src/rechunk_v2.py：請從數位學習平台下載，放到專案的 src/ 資料夾"
raw_files = sorted((ROOT / "data" / "interim").glob("*/raw_*.md"))
assert raw_files, "data/interim/ 中沒有 raw_*.md：請下載 week12_interim_raw.zip 解壓縮到 data/interim/"
assert (ROOT / "models" / "bge-small-zh-v1.5" / "config.json").exists(), "找不到 models/bge-small-zh-v1.5/：請回到第 10 週的 10_0_setup 步驟 2"

sys.path.insert(0, str(ROOT / "src"))   # 讓 Python 找得到 src/rechunk_v2.py
import rechunk_v2
print(f"✅ 找到 rechunk_v2.py、{len(raw_files)} 份 raw_*.md、句向量模型")

專案根目錄： c:\Users\howar\Desktop\global_languages
✅ 找到 rechunk_v2.py、9 份 raw_*.md、句向量模型


In [2]:
# 重新切塊的結果不上傳 GitHub（和第 8 週的 data/processed/ 一樣）：自動加進 .gitignore
gi = ROOT / ".gitignore"
text = gi.read_text(encoding="utf-8") if gi.exists() else ""
need = ["data/processed_v2/", "data/processed_v2_800/", "data/embeddings/"]
added = [line for line in need if line not in text.splitlines()]
if added:
    gi.write_text(text.rstrip("\n") + "\n\n# Week 12\n" + "\n".join(added) + "\n", encoding="utf-8")
print("✅ .gitignore 已包含：", need, "（這次新增：", added or "無", "）")

✅ .gitignore 已包含： ['data/processed_v2/', 'data/processed_v2_800/', 'data/embeddings/'] （這次新增： 無 ）


---
# 第 1 節｜為什麼要重新切塊 (9:00–9:50)

## 講解 1-1｜第 8 週的 chunks 有什麼問題？

第 8 週只在 `##` 標題處切開。Docling 把很多條文輸出成清單項目（`- 第十九條 …`），不是標題，所以**兩個 `##` 之間的整章、甚至整份法規，變成一個 chunk**。下一格看看長度分布。

In [3]:
# 讀取第 8 週的 chunks（順序和第 10 週相同，才能沿用第 10 週算好的向量）
W8_DIR = ROOT / "data" / "processed"
w8 = []
for path in sorted(W8_DIR.glob("*/cleaned_*_processed.json")):
    for c in json.loads(path.read_text(encoding="utf-8")):
        w8.append({"text": c["text"], "embed_text": c["text"], "source_file": c["metadata"]["source_file"]})
w8_len = pd.Series([len(c["text"]) for c in w8])
print(f"第 8 週：{len(w8)} 個 chunks｜中位數 {int(w8_len.median())} 字｜最長 {int(w8_len.max())} 字｜超過 1000 字：{int((w8_len > 1000).sum())} 個")

第 8 週：350 個 chunks｜中位數 269 字｜最長 14335 字｜超過 1000 字：79 個


## 講解 1-2｜為什麼模型只讀 512 個 token？（只講授，不實作）

句向量模型是 Transformer。Transformer 的 Attention：**每一個字都要和其他每一個字比較一次**。

| chunk 長度 | 要比較的次數 |
|---|---|
| 512 字 | 512 × 512 ≈ **26 萬** |
| 14,350 字 | 14,350 × 14,350 ≈ **2 億** |

長度變 28 倍，計算量變約 780 倍。所以模型設定一個上限：`bge-small-zh-v1.5` 一次最多讀 **512 個 token**（中文大約一個字一個 token），**超過的部分直接丟掉**。

→ 一個 7,000 字的 chunk，模型只看到前面約 500 字，後面的條文對語意檢索來說「不存在」。

## 講解 1-3｜四種切塊策略

**A. 依標題切（第 8 週）**：只在 `##` 切。標題剛好對應條文時很好；Docling 沒有把條文標成 `##` 時，整章變成一塊。

**B. 固定長度 + 重疊**：不管內容，每 N 個字切一刀。最簡單、最常用的基準方法。缺點：可能從一句話的中間切開。所以讓相鄰兩塊**重疊**一段：一句話如果被切斷，至少會完整出現在其中一塊。

```
文字：  ├──────── 500 ────────┤
              ├──────── 500 ────────┤        每次往後走 500 − 100 = 400 字
                    ├──────── 500 ────────┤  重疊 100 字
```

**C. 依文件結構切**：法規本身就有結構：「第N條」是一個完整的意思單位。每一條一塊；太長的條文再依「一、→（一）→ 1.」切開（**遞迴切分**），每塊都帶著這一條的第一句。

**D. C ＋ 上下文標題**：每塊開頭加「【法規名稱｜篇｜章】第N條」。「第三條 申請期限」單獨看不知道是哪一份法規；加上標題，模型就知道。

另外，C、D 都用了一個技巧：**算向量的文字和交給語言模型的文字可以不一樣**。`embed_text` 只放中文（模型讀得完），`text` 保留中英對照（第 14 週給語言模型看）。

C 和 D 由老師的程式 `src/rechunk_v2.py` 產生；B 由你在任務 1-B 寫出來。

**「清理過的文字」是什麼？** `rechunk_v2.py` 在切塊之前，先對 Docling 的原始輸出 (`raw_*.md`) 做清理：刪頁首頁尾、頁碼、目錄、換頁時插入的修訂日期；把 PDF 斷行造成的「課 程」接回「課程」；把被拆成兩個清單項目的句子接回。做法和你第 8 週練習的正則清理是同一類，這次由老師寫好，**直接使用，不需要讀懂**。

| | 清理 | 切法 |
|---|---|---|
| A | 你自己第 8 週的正則 | 依 `##` 標題 |
| B、C、D | 老師的 `rechunk_v2.py`（**三者相同**） | 各不相同 |

所以：**B、C、D 之間的比較，只差「切法」**；A 和其他三者比，清理和切法**兩者都不同**（A 是「之前」的基準）。

## 任務 1-A（教師示範）：執行依結構切塊（C、D）

在 VS Code 終端機（確認有 `(.venv)`，位置在**專案根目錄**）執行：

```
python src/rechunk_v2.py
```

看到「讀取 9 份法規 → 264 個 chunks」之類的訊息後，執行下一格讀取結果。（也可以直接執行下一格：如果還沒有切過，它會幫你切。）

以下程式中，`v2` 就是策略 D 的 chunks；策略 C 用同一批 chunks，只是算向量時拿掉第一行的標題。

In [4]:
V2_DIR = ROOT / "data" / "processed_v2"
if not list(V2_DIR.glob("*/v2_*.json")):
    print("還沒有 data/processed_v2/，現在執行 rechunk_v2 …")
    rechunk_v2.rechunk_all(ROOT / "data" / "interim", V2_DIR)

def load_v2(folder):
    chunks = []
    for path in sorted(Path(folder).glob("*/v2_*.json")):
        chunks += json.loads(path.read_text(encoding="utf-8"))
    return chunks

v2 = load_v2(V2_DIR)
v2_len = pd.Series([c["metadata"]["embed_length"] for c in v2])
print(f"重新切塊：{len(v2)} 個 chunks｜中文部分中位數 {int(v2_len.median())} 字｜最長 {int(v2_len.max())} 字")

# 每份法規：找到幾條、切成幾塊
table = (pd.DataFrame([c["metadata"] for c in v2])
         .query("chunk_type in ['article', 'article_en']")
         .groupby("doc_title").agg(條數=("article_no", "nunique"), 最大條號=("article_no", "max"), chunks=("chunk_id", "count")))
table

重新切塊：264 個 chunks｜中文部分中位數 160 字｜最長 1522 字


,條數,最大條號,chunks
doc_title,,,
圖書館管理作業規範,53,53.0,73
大學部學則,83,83.0,83
學生宿舍學術網路使用及管理辦法,9,9.0,9
學生抵免學分辦法,11,11.0,12
學生機車管理辦法,7,7.0,9
學生獎懲規定,19,19.0,23
學生申請期中停修課程實施要點,9,9.0,9
學生請假作業辦法,7,7.0,9
學生電子郵件帳號管理要點,8,8.0,8


In [5]:
# 看兩個例子（策略 D）：短條文一塊；長條文切成好幾塊，每塊都帶著第一句
def show(doc_title, article_no, width=300):
    for c in v2:
        m = c["metadata"]
        if m["doc_title"] == doc_title and m["article_no"] == article_no:
            print(f"── {m['chunk_id']}（中文 {m['embed_length']} 字）")
            print(c["embed_text"][:width], "\n")

show("學生申請期中停修課程實施要點", 2)
show("學生獎懲規定", 10, width=160)

── A231060004#2（中文 92 字）
【學生申請期中停修課程實施要點】第二條 適用範圍
每學期申請停修科目以2科為限只限於選修課，惟停修後之總學分數不得少於當學期應修最低學分數；延長修業年限學生停修後至少仍應修習一個科目。 

── A242030012#13（中文 491 字）
【學生獎懲規定】第十條
凡有下列各款情形之一者，予以大過處分。
一、冒用、假借、塗改、偽造文書及證件者。
二、借與他人使用己有證件致妨礙他人權益者。
三、冒用師生帳號密碼，破解使用電腦之保護措施，或篡改學校系統資料者。
四、利用電腦網路散佈不實謠言、不雅文字、圖片，而污辱損人名譽者。
五、侵犯他人智慧財産權，經媒體及司 

── A242030012#14（中文 234 字）
【學生獎懲規定】第十條
凡有下列各款情形之一者，予以大過處分。
十七、未遵守學校防疫措施，造成他人、學校嚴重損失者。
十八、未經允許以不當方式(跟車或偽造車牌)進入校園或停車場者。
十九、凡請人代寫或代寫他人報告或論文者。
二十、違反學生生活輔導管理辦法中住宿生活規定或不遵守生活輔導組公告之相關禁止事項，影響公共安全或 



## 任務 1-B（兩人一組）：寫出策略 B「固定長度 + 重疊」

完成 `fixed_chunks(text, size, overlap)`：從第 0 個字開始，每次往後走 `size - overlap` 個字，每次取 `size` 個字。

提示：
- 每一塊是 `text[i : i + size]`
- `i` 從 0 開始，每次加 `size - overlap`：`range(0, len(text), size - overlap)`
- 一行就可以完成：`[... for i in range(...)]`

In [6]:
def fixed_chunks(text, size=500, overlap=100):

    if size <= 0 or overlap < 0 or overlap >= size:

        raise ValueError("size 必須大於 0，且 overlap 必須介於 0（含）與 size（不含）之間")

    step = size - overlap

    return [text[i:i + size] for i in range(0, len(text), step)]


In [7]:
# 檢查 TODO 1
assert fixed_chunks("abc", 2, 0) is not ..., "TODO 1 還沒完成：把 ... 換成你的程式"
assert fixed_chunks("abcdefghij", size=4, overlap=1) == ["abcd", "defg", "ghij", "j"], \
    "「abcdefghij」每塊 4 個字、重疊 1 個字，應該是 ['abcd', 'defg', 'ghij', 'j']：每次往後走 4 - 1 = 3 個字"
assert fixed_chunks("abcdef", size=3, overlap=0) == ["abc", "def"], "overlap = 0 時就是單純每 3 個字切一刀"
print("✅ 1-B 完成")

✅ 1-B 完成


In [8]:
# 用你的 fixed_chunks 產生策略 B：和 C、D 一樣，從清理過的 Docling 文字開始
SIZE, OVERLAP = 500, 100
fixed = []
for path in raw_files:
    code_, title = rechunk_v2.doc_info(path)
    text = "\n".join(rechunk_v2.clean_lines(path.read_text(encoding="utf-8"), title))
    for k, piece in enumerate(fixed_chunks(text, SIZE, OVERLAP)):
        fixed.append({"text": piece, "embed_text": piece, "chunk_id": f"{code_}#fixed{k}"})
print(f"策略 B：{len(fixed)} 個 chunks（每塊 {SIZE} 字、重疊 {OVERLAP} 字）")

def norm(text):
    return re.sub(r"\s+", "", text)

def find_chunks(chunks, phrase):
    """回傳 text 中含有 phrase 的 chunks（比對前去掉所有空白）"""
    return [c for c in chunks if norm(phrase) in norm(c["text"])]

for phrase in ["停修科目以2科為限", "各種考試舞弊者", "學生每學期修習學分數不得多於廿七學分"]:
    a = [len(c["text"]) for c in find_chunks(w8, phrase)]
    b = [len(c["text"]) for c in find_chunks(fixed, phrase)]
    d = [c["metadata"]["embed_length"] for c in find_chunks(v2, phrase)]
    print(f"「{phrase}」\n   A 第8週：{a} 字   B 固定長度：{b} 字   D 依條文（中文）：{d} 字")

策略 B：808 個 chunks（每塊 500 字、重疊 100 字）
「停修科目以2科為限」
   A 第8週：[82] 字   B 固定長度：[500] 字   D 依條文（中文）：[92] 字
「各種考試舞弊者」
   A 第8週：[7491] 字   B 固定長度：[500] 字   D 依條文（中文）：[491] 字
「學生每學期修習學分數不得多於廿七學分」
   A 第8週：[7204] 字   B 固定長度：[500] 字   D 依條文（中文）：[381] 字


**討論：**
1. 「各種考試舞弊者」在第 8 週的 chunk 有幾千字。這句話在那個 chunk 的前 500 字裡面嗎？模型讀得到嗎？
2. 策略 B 中，有些句子出現在**兩個** chunk 裡。為什麼？（提示：重疊）
3. 隨便印一個 `fixed[10]["text"]` 看看：開頭和結尾是完整的句子嗎？

## 第 1 節小結

- 第 8 週只在 `##` 切，造成很多超長 chunk；Transformer 的計算量隨長度快速增加，模型只讀前 512 個 token。
- 四種切塊策略：依標題（A）、固定長度 + 重疊（B）、依文件結構（C）、結構 + 上下文標題（D）。
- 固定長度最簡單，但會切斷句子；重疊讓被切斷的句子至少完整出現在一塊中。

---
# 第 2 節｜向量索引與自動評估 (10:00–10:50)

## 講解 2-1｜什麼是向量索引？

第 10 週的語意檢索：每次都把所有 chunks 變成向量，再和問題比較。chunks 不會常常改變，**向量只要算一次、存起來**，之後每次只算問題的向量就好。存起來的東西就叫**向量索引 (vector index)**：

| 檔案 | 內容 |
|---|---|
| `data/embeddings/week12_v2_emb.npy` | 一個矩陣：每一列是一個 chunk 的向量（264 列 × 512 欄） |
| `data/embeddings/week12_v2_ids.json` | 第幾列對應哪一個 chunk |

搜尋 = 一次矩陣乘法（和第 10 週的 `chunk_emb @ q` 一樣）。chunks 很多的時候（幾百萬個），會改用 FAISS、Chroma 這類專門的工具，原理相同。

## 任務 2-A（教師示範）：建立向量索引

先建立策略 D 的索引（第 14 週要用的就是這一個）。在 CPU 上約需 30 秒到 2 分鐘。第 8 週（A）的向量沿用第 10 週算好的檔案。

In [9]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(str(ROOT / "models" / "bge-small-zh-v1.5"))

def encode(texts):
    return model.encode(texts, normalize_embeddings=True, batch_size=16, show_progress_bar=True)

EMB_DIR = ROOT / "data" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

# 策略 D 的索引：用 embed_text（只有中文）算向量
t0 = time.time()
v2_emb = encode([c["embed_text"] for c in v2])
np.save(EMB_DIR / "week12_v2_emb.npy", v2_emb)
(EMB_DIR / "week12_v2_ids.json").write_text(json.dumps([c["metadata"]["chunk_id"] for c in v2], ensure_ascii=False), encoding="utf-8")
print(f"✅ 向量索引已儲存：{v2_emb.shape}，{time.time() - t0:.0f} 秒")

# 策略 A（第 8 週）：沿用第 10 週的向量（數量不同就重新計算）
W10_EMB = EMB_DIR / "week10_chunk_emb.npy"
if W10_EMB.exists() and np.load(W10_EMB).shape[0] == len(w8):
    w8_emb = np.load(W10_EMB)
    print("第 8 週向量：沿用", W10_EMB.name)
else:
    w8_emb = encode([c["text"] for c in w8])
    print("第 8 週向量：重新計算完成")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

✅ 向量索引已儲存：(264, 512)，1 秒
第 8 週向量：沿用 week10_chunk_emb.npy


## 講解 2-2｜怎麼自動判斷「找對了沒」？

第 10 週是人工判斷。今天改成**問答集 (question set)**：每一題附上一段「答案片段」（能回答這一題的原文）。

- 檢索結果的第 *r* 名 chunk 含有答案片段 → 這一題的**排名 (rank)** 是 *r*
- **hit@k**：排名 ≤ k 的題目比例。k = 1、3、5 各算一次
- 用「答案片段」而不是 chunk 編號：**第 8 週和重新切塊的 chunks 編號不同，但答案片段一樣**，所以兩邊可以公平比較

問答集共 24 題：Q01–Q10 是第 10 週的 10 題（單數題用詞和條文相同、雙數題換句話說），N01–N14 是新題。每題都對得到一份法規的某一條。

In [10]:
QSET = [
    ("Q01", "每學期申請停修科目以幾科為限？", "停修科目以2科為限"),
    ("Q02", "這學期課太重，想中途放棄一門課，最多可以放棄幾門？", "停修科目以2科為限"),
    ("Q03", "大學部學生借書期限及冊數為何？", "大學部、專科部學生借書含附件共以二十冊為限"),
    ("Q04", "我跟圖書館借的書可以放在我這邊多久？", "大學部、專科部學生借書含附件共以二十冊為限"),
    ("Q05", "借書逾期罰款每日多少元？", "處以每冊（件）每日罰款5元"),
    ("Q06", "書太晚還會被罰錢嗎？", "處以每冊（件）每日罰款5元"),
    ("Q07", "請病假應檢附什麼證明？", "一般病假：1天假應有就醫證明"),
    ("Q08", "我感冒發燒沒去上課，事後要補交什麼文件？", "一般病假：1天假應有就醫證明"),
    ("Q09", "考試舞弊者予以何種處分？", "各種考試舞弊者"),
    ("Q10", "考試偷看別人答案被抓到會怎樣？", "各種考試舞弊者"),
    ("N01", "每學期最多可以修幾學分？", "學生每學期修習學分數不得多於廿七學分"),
    ("N02", "大學部畢業至少要修滿幾學分？", "至少須修滿一百四十八學分"),
    ("N03", "一學期曠課達幾小時會被退學？", "全學期曠課達四十五小時者"),
    ("N04", "休學最多可以休幾年？", "休學累計以不超過二學年為原則"),
    ("N05", "機車停車證一學年的清潔費是多少錢？", "機車停放清潔費每學年為500元"),
    ("N06", "機車停車證弄丟了，補發要繳多少錢？", "工本費100元"),
    ("N07", "宿舍網路每天的流量上限是多少？", "總流量為8GB"),
    ("N08", "心理健康假一學期最多可以請幾天？", "每學期最多7天"),
    ("N09", "病假如果太晚補請會怎樣？", "第7日至第12日補辦請假手續須記申誡1次"),
    ("N10", "在宿舍偷偷養貓會被怎麼處罰？", "未經允許在宿舍飼養寵物者"),
    ("N11", "抵免學分最多可以抵多少？", "抵免總學分數不得超過各系規定最低畢業學分三分之一"),
    ("N12", "圖書館團體討論室最少要幾個人才能借？", "討論室需最少三人使用方得提出申請借用"),
    ("N13", "學校的電子郵件帳號是用什麼當帳號？", "即以其學號為電子郵件帳號統一建立"),
    ("N14", "期末考那幾週自習室開到幾點？", "考試週及前一週（考試週期）每天24小時開放"),
]
pd.DataFrame(QSET, columns=["編號", "問題", "答案片段"])

,編號,問題,答案片段
0,Q01,每學期申請停修科目以幾科為限？,停修科目以2科為限
1,Q02,這學期課太重，想中途放棄一門課，最多可以放棄幾門？,停修科目以2科為限
2,Q03,大學部學生借書期限及冊數為何？,大學部、專科部學生借書含附件共以二十冊為限
3,Q04,我跟圖書館借的書可以放在我這邊多久？,大學部、專科部學生借書含附件共以二十冊為限
4,Q05,借書逾期罰款每日多少元？,處以每冊（件）每日罰款5元
5,Q06,書太晚還會被罰錢嗎？,處以每冊（件）每日罰款5元
6,Q07,請病假應檢附什麼證明？,一般病假：1天假應有就醫證明
7,Q08,我感冒發燒沒去上課，事後要補交什麼文件？,一般病假：1天假應有就醫證明
8,Q09,考試舞弊者予以何種處分？,各種考試舞弊者
9,Q10,考試偷看別人答案被抓到會怎樣？,各種考試舞弊者


In [11]:
def search(emb, query, k=5):
    """回傳前 k 名的列號（和第 10 週的 search_dense 相同的做法）"""
    q = model.encode([query], normalize_embeddings=True)[0]
    return list(np.argsort(-(emb @ q))[:k])

def gold_rank(chunks, emb, query, evidence):
    """正確答案排第幾名：第一個含有答案片段的 chunk 的名次（找不到 → 999）"""
    q = model.encode([query], normalize_embeddings=True)[0]
    for rank, i in enumerate(np.argsort(-(emb @ q)), 1):
        if norm(evidence) in norm(chunks[i]["text"]):
            return rank
    return 999

ranks = {
    "A 第8週": [gold_rank(w8, w8_emb, q, ev) for _, q, ev in QSET],
    "D 依條文+標題": [gold_rank(v2, v2_emb, q, ev) for _, q, ev in QSET],
}
pd.DataFrame(ranks, index=[x[0] for x in QSET]).T

,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,Q09,Q10,...,N05,N06,N07,N08,N09,N10,N11,N12,N13,N14
A 第8週,1,1,1,7,1,1,1,21,1,1,...,1,1,1,1,4,127,2,1,2,1
D 依條文+標題,1,1,13,4,1,1,1,84,2,4,...,1,1,1,1,4,3,6,1,4,1


## 任務 2-B（兩人一組）：計算 hit@k 和「前 k 名的總字數」

1. `hit_at_k(rank_list, k)`：排名 ≤ k 的題目比例。提示：`np.mean([...])`
2. `topk_chars(chunks, emb, query, k)`：前 k 名 chunks 的 `embed_text` 總字數。提示：用上面的 `search(...)` 取得前 k 名的列號

**為什麼要看總字數？** 第 14 週要把前 k 名 chunks 一起交給語言模型（送的是 `embed_text`：只有中文）。字數越多，越慢、越貴；免費帳號每分鐘只能處理約 8,000 個 token。

In [12]:
def hit_at_k(rank_list, k):

    return float(np.mean([rank <= k for rank in rank_list])) if rank_list else 0.0



def topk_chars(chunks, emb, query, k):

    return sum(len(chunks[i]["embed_text"]) for i in search(emb, query, k))


In [13]:
# 檢查 TODO 2、3
assert hit_at_k([1], 1) is not ..., "TODO 2 還沒完成：把 ... 換成你的程式"
assert topk_chars(v2, v2_emb, QSET[0][1], 1) is not ..., "TODO 3 還沒完成：把 ... 換成你的程式"
assert abs(hit_at_k([1, 4, 2, 10], 3) - 0.5) < 1e-9, "[1, 4, 2, 10] 中排名 <= 3 的有 2 題，hit@3 應該是 0.5"
assert hit_at_k([1, 1], 1) == 1.0 and hit_at_k([999], 5) == 0.0, "請再檢查 hit_at_k"
_q = QSET[0][1]
assert topk_chars(v2, v2_emb, _q, 3) == sum(len(v2[i]["embed_text"]) for i in search(v2_emb, _q, 3)), "topk_chars 應該加總前 k 名的 len(embed_text)"
print("✅ 2-B 完成")

✅ 2-B 完成


In [14]:
def evaluate(name, chunks, emb, rank_list):
    sizes3 = [topk_chars(chunks, emb, q, 3) for _, q, _ in QSET]
    sizes5 = [topk_chars(chunks, emb, q, 5) for _, q, _ in QSET]
    return {"版本": name, "chunks": len(chunks),
            "hit@1": round(hit_at_k(rank_list, 1), 2), "hit@3": round(hit_at_k(rank_list, 3), 2), "hit@5": round(hit_at_k(rank_list, 5), 2),
            "前3名總字數(中位數)": int(np.median(sizes3)), "前3名總字數(最大)": int(max(sizes3)),
            "前5名總字數(最大)": int(max(sizes5))}

results = [evaluate("A 第8週", w8, w8_emb, ranks["A 第8週"]),
           evaluate("D 依條文+標題", v2, v2_emb, ranks["D 依條文+標題"])]
pd.DataFrame(results).set_index("版本")

,chunks,hit@1,hit@3,hit@5,前3名總字數(中位數),前3名總字數(最大),前5名總字數(最大)
版本,,,,,,,
A 第8週,350,0.67,0.75,0.79,756,12681,20024
D 依條文+標題,264,0.54,0.71,0.88,709,1197,1832


**觀察（和隔壁同學討論）：**

1. A 和 D：hit@3 誰比較高？hit@5 呢？
2. 「前 3 名總字數（最大）」差多少？如果要把前 5 名交給語言模型，哪一個放得進每分鐘 8,000 token？
3. 看上面的排名表：A 有沒有哪一題排到幾十名以後？D 呢？

## 第 2 節小結

- 向量索引 = chunks 的向量算一次、存起來；搜尋只要算問題的向量。
- 用「問答集 + 答案片段」自動計算排名與 hit@k，第 8 週和重新切塊可以公平比較。
- 評估檢索不只看 hit@k，也要看**送給語言模型的字數**。

---
# 第 3 節｜四種策略一起比較 (11:00–11:50)

## 任務 3-A（教師示範）：加入策略 B、C

- **B**：你寫的固定長度 + 重疊。chunks 比較多，CPU 約需 1–3 分鐘。
- **C**：和 D 同一批 chunks，算向量時**拿掉第一行的【…】標題**。C 和 D 只差標題這一件事，所以 C vs D 就是在測「上下文標題有沒有用」。

In [15]:
fixed_emb = encode([c["embed_text"] for c in fixed])

def without_header(text):
    lines = text.splitlines()
    return "\n".join(lines[1:]) if len(lines) > 1 and lines[0].startswith("【") else text

c_emb = encode([without_header(c["embed_text"]) for c in v2])

ranks["B 固定長度+重疊"] = [gold_rank(fixed, fixed_emb, q, ev) for _, q, ev in QSET]
ranks["C 依條文(無標題)"] = [gold_rank(v2, c_emb, q, ev) for _, q, ev in QSET]
results.insert(1, evaluate("B 固定長度+重疊", fixed, fixed_emb, ranks["B 固定長度+重疊"]))
results.insert(2, evaluate("C 依條文(無標題)", v2, c_emb, ranks["C 依條文(無標題)"]))

display(pd.DataFrame(results).set_index("版本"))
pd.DataFrame(ranks, index=[x[0] for x in QSET]).T[[x[0] for x in QSET]]

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

,chunks,hit@1,hit@3,hit@5,前3名總字數(中位數),前3名總字數(最大),前5名總字數(最大)
版本,,,,,,,
A 第8週,350,0.67,0.75,0.79,756,12681,20024
B 固定長度+重疊,808,0.38,0.67,0.79,1500,1500,2500
C 依條文(無標題),264,0.50,0.67,0.83,708,1197,2015
D 依條文+標題,264,0.54,0.71,0.88,709,1197,1832


,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,Q09,Q10,...,N05,N06,N07,N08,N09,N10,N11,N12,N13,N14
A 第8週,1,1,1,7,1,1,1,21,1,1,...,1,1,1,1,4,127,2,1,2,1
D 依條文+標題,1,1,13,4,1,1,1,84,2,4,...,1,1,1,1,4,3,6,1,4,1
B 固定長度+重疊,2,7,1,2,2,6,1,7,9,5,...,2,2,1,1,3,3,5,1,4,1
C 依條文(無標題),1,5,2,8,1,1,1,48,9,8,...,1,1,1,1,5,4,4,1,3,1


**討論（和隔壁同學）：**

（提醒：B、C、D 的清理相同，只差切法；A 的清理是你自己第 8 週的，所以 A 和其他三者的差異，不能只歸因於切法。）

1. 四種策略中，hit@5 最高的是哪一個？「前 5 名總字數」最少的呢？兩者是同一個嗎？
2. **B vs C**：都是短 chunk，差別是「不管內容切」還是「依條文切」。結果差多少？
3. **C vs D**：只差標題列。標題列有幫助嗎？哪幾題差最多？
4. 排名表中，哪一題四種策略都找不到（排名都很後面）？你覺得為什麼？

## 講解 3-2｜24 題能下結論嗎？



24 題中，**1 題 = 4%**。hit@3 差 0.04 只差 1 題，可能只是運氣（剛好那一題換了說法）。



所以：差異小的時候，**看排名表找原因**（哪幾題變好、哪幾題變差、為什麼），不要只看一個數字。題目越多，結論越可靠。



## 任務 3-C（個人）：自己出 2 題



1. 打開 `data/processed_v2/` 裡任一份 JSON，找一條**你覺得同學可能會問**的條文。

2. 寫一個問題，並從那一條**原封不動複製**一小段當作「答案片段」（10–30 字）。

3. 第 1 題用詞和條文接近；第 2 題用口語、換句話說。

4. 執行後看看 A、B、D 各排第幾名，在 `why` 寫一兩句觀察。



**不可以和問答集的 24 題相同。**


In [16]:
# TODO 4：填入你的 2 個問題

my_questions = [

    {"question": "下學期申辦機車停車證的清潔費是多少？", "evidence": "申辦「機車停車證」者，收場地清潔費300元", "style": "用詞接近條文", "why": "A 排第 2，B、D 都排第 1；固定長度與加上條文標題的策略都比第 8 週更早找到答案。"},

    {"question": "我下學期才要辦機車證，要多付多少場地費？", "evidence": "申辦「機車停車證」者，收場地清潔費300元", "style": "口語、換句話說", "why": "A 排第 2，B 和 D 都排第 1，口語改寫題也由兩種新切塊方式把答案排在最前面。"},

]



for item in my_questions:

    assert item["question"].strip() and item["evidence"].strip(), "請填入問題和答案片段"

    assert item["question"] not in [q for _, q, _ in QSET], "不可以和問答集的題目相同"

    assert find_chunks(v2, item["evidence"]), f"在重新切塊的 chunks 中找不到「{item['evidence']}」：請從 JSON 的 text 原封不動複製"



for item in my_questions:

    item["rank"] = {

        "A 第8週": gold_rank(w8, w8_emb, item["question"], item["evidence"]),

        "B 固定長度+重疊": gold_rank(fixed, fixed_emb, item["question"], item["evidence"]),

        "D 依條文+標題": gold_rank(v2, v2_emb, item["question"], item["evidence"]),

    }

    print(item["question"], "→", item["rank"])

print("✅ 3-C 的題目都找得到答案片段。請在上面的 why 寫下觀察，再執行一次這一格")


下學期申辦機車停車證的清潔費是多少？ → {'A 第8週': 2, 'B 固定長度+重疊': 1, 'D 依條文+標題': 1}
我下學期才要辦機車證，要多付多少場地費？ → {'A 第8週': 2, 'B 固定長度+重疊': 1, 'D 依條文+標題': 1}
✅ 3-C 的題目都找得到答案片段。請在上面的 why 寫下觀察，再執行一次這一格


### 反思（請直接在這一格中作答，每題 2–3 句）

1. 四種切塊策略中，你會選哪一個給期末系統？請用 hit@k 和「前 k 名總字數」兩種數字說明。

   > 我會選 D（依條文加標題）：四種策略中 D 的 hit@5 最高，為 0.88，hit@3 為 0.71；前 5 名 chunks 的最大字數是 1,832，低於 8,000 字。A 的 hit@3 是 0.75，但前 5 名最多達 20,024 字，檢索上下文較長。

2. 固定長度 + 重疊（B）簡單又不需要知道文件結構。在什麼情況下你會選它，而不是依結構切？（提示：如果文件不是法規，沒有「第N條」呢？）

   > 如果文件沒有穩定的章節或條文標記，例如一般文章、訪談逐字稿或混合格式資料，我會選固定長度加重疊作為容易套用的基準。這次把 B 從 500/100 改成 300/50 後，chunks 從 808 增至 1,292，hit@3 維持 0.67、hit@5 從 0.79 降到 0.75，但前 5 名最大字數從 2,500 降至 1,500；短 chunks 減少上下文長度，命中率也有取捨。

3. 你自己出的口語題，排名如何？如果找不到，你覺得還可以怎麼改進檢索？（提示：第 10 週講過的「混合檢索」）

   > 我出的口語題在 A 排第 2，在 B（500/100）、B（300/50）和 D 都排第 1，兩種固定長度設定和結構切塊都找到了答案。兩題的結果相同，但題數有限；之後我會增加不同主題與問法，並在排名較差時測試混合關鍵字與向量檢索。

### 選做：改一個設定

一次只改一個設定，其他都不變，才知道差異是哪個設定造成的。例如：
- 策略 B：`SIZE, OVERLAP = 300, 50` 或 `800, 150`
- 策略 D：每塊最多 800 字（下一格，結果存到 `data/processed_v2_800/`，不會覆蓋原本的）

In [17]:
RUN_OPTIONAL = True   # 策略 B 改用 size=300、overlap=50；原始 500/100 結果保留比較
if RUN_OPTIONAL:
    SIZE_ALT, OVERLAP_ALT = 300, 50
    fixed_alt = []
    for path in raw_files:
        code_, title = rechunk_v2.doc_info(path)
        text = "\n".join(rechunk_v2.clean_lines(path.read_text(encoding="utf-8"), title))
        for k, piece in enumerate(fixed_chunks(text, SIZE_ALT, OVERLAP_ALT)):
            fixed_alt.append({"text": piece, "embed_text": piece, "chunk_id": f"{code_}#fixed300_50_{k}"})
    fixed_alt_emb = encode([c["embed_text"] for c in fixed_alt])
    alt_name = "B 固定長度+重疊 (300/50)"
    ranks[alt_name] = [gold_rank(fixed_alt, fixed_alt_emb, q, ev) for _, q, ev in QSET]
    results.append(evaluate(alt_name, fixed_alt, fixed_alt_emb, ranks[alt_name]))
    display(pd.DataFrame(results).set_index("版本"))

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

,chunks,hit@1,hit@3,hit@5,前3名總字數(中位數),前3名總字數(最大),前5名總字數(最大)
版本,,,,,,,
A 第8週,350,0.67,0.75,0.79,756,12681,20024
B 固定長度+重疊,808,0.38,0.67,0.79,1500,1500,2500
C 依條文(無標題),264,0.50,0.67,0.83,708,1197,2015
D 依條文+標題,264,0.54,0.71,0.88,709,1197,1832
B 固定長度+重疊 (300/50),1292,0.25,0.67,0.75,900,900,1500


## 繳交

執行下一格，產生 `week12_retrieval_eval.json`。

In [18]:
for item in my_questions:

    assert item["why"].strip(), "請在 my_questions 的 why 寫下觀察"



out = {

    "week": 12,

    "model": "BAAI/bge-small-zh-v1.5",

    "n_chunks": {r["版本"]: r["chunks"] for r in results},

    "results": results,

    "ranks": {name: dict(zip([x[0] for x in QSET], [int(r) for r in rs])) for name, rs in ranks.items()},

    "my_questions": my_questions,

}

OUT_PATH = ROOT / "week12_retrieval_eval.json"

OUT_PATH.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

print("已儲存：", OUT_PATH)


已儲存： c:\Users\howar\Desktop\global_languages\week12_retrieval_eval.json


### 更新 README.md

Week 12 任務清單與評估結果已整理於專案根目錄 `README.md`。

### 下課前檢查清單

- [x] 所有 `✅` 檢查都通過，notebook **保留輸出**並存檔
- [x] 反思 3 題已作答
- [x] `week12_retrieval_eval.json` 已產生
- [x] `src/rechunk_v2.py` 在專案裡
- [x] `.gitignore` 已更新（忽略 `data/processed_v2/`、`data/processed_v2_800/` 和 embeddings）
- [x] `README.md` 已新增 **Week 12** 任務清單與 hit@k 表格
- [x] `AI_USAGE.md` 已新增 **Week 12** 段落
- [x] (選做) 策略 B 改成 `300/50` 並比較
- [ ] commit & push，並**打開 GitHub 網頁確認**

```bash
git add notebooks/12_vector_indexing.ipynb src/rechunk_v2.py week12_retrieval_eval.json README.md AI_USAGE.md .gitignore
git commit -m "Week 12: re-chunking and vector index"
git push
```
